# Some Statistics of OpenAIRE

## 1. Get number of papers and schema of the dump

In [1]:
from pathlib import Path
import polars as pl
import pyarrow.parquet as pq
from tqdm import tqdm
from collections import Counter, defaultdict

In [2]:
parquet_dir = Path("/home/jarenas/Datasets/OpenAIRE/20250113")
parquet_files = sorted(parquet_dir.glob("*.parquet"))

total_rows = 0

for p in tqdm(parquet_files, desc="Counting rows via PyArrow metadata"):
    pf = pq.ParquetFile(p)
    total_rows += pf.metadata.num_rows

print("Total rows (OpenAIRE dump):", total_rows)

Counting rows via PyArrow metadata: 100%|██████████████████████████████████████████████████████████| 800/800 [00:00<00:00, 5910.15it/s]

Total rows (OpenAIRE dump): 140306502


In [3]:
def print_schema_tree(schema, indent: str = "", is_last: bool = True):
    """
    Imprime un schema de Polars en formato árbol legible.
    Compatible con Structs y Lists de Structs.
    """
    # Schema: dict-like {name: dtype}
    if hasattr(schema, "items"):
        items = list(schema.items())
    else:
        # Lista de Field (Struct.fields)
        items = [(f.name, f.dtype) for f in schema]

    for i, (name, dtype) in enumerate(items):
        last = i == len(items) - 1
        branch = "└── " if last else "├── "
        print(f"{indent}{branch}{name}: {dtype}")

        # Struct
        if isinstance(dtype, pl.Struct):
            new_indent = indent + ("    " if last else "│   ")
            print_schema_tree(dtype.fields, new_indent, last)

        # List of Struct
        elif isinstance(dtype, pl.List) and isinstance(dtype.inner, pl.Struct):
            new_indent = indent + ("    " if last else "│   ")
            print_schema_tree(dtype.inner.fields, new_indent, last)

In [4]:
sample_path = next(parquet_dir.glob("*.parquet"))
df_sample = pl.read_parquet(sample_path)

print("Schema OpenAIRE (árbol):")
print_schema_tree(df_sample.schema)

Schema OpenAIRE (árbol):
├── id: String
├── date: String
├── year: Int32
├── access_mode: String
├── titles: List(String)
├── peer_reviewed: Boolean
├── abstracts: List(String)
├── affiliations: List(Struct({'id': String, 'name': String, 'country': Struct({'code': String, 'name': String, 'continent': String})}))
│   ├── id: String
│   ├── name: String
│   └── country: Struct({'code': String, 'name': String, 'continent': String})
│       ├── code: String
│       ├── name: String
│       └── continent: String
├── country_groups: List(String)
├── fos: List(Struct({'lvl1': String, 'lvl2': String, 'lvl3': String, 'lvl4': String}))
│   ├── lvl1: String
│   ├── lvl2: String
│   ├── lvl3: String
│   └── lvl4: String
├── sdg: List(String)
├── pids: List(Struct({'type': String, 'pid': String}))
│   ├── type: String
│   └── pid: String
├── citations: Struct({'total': Int64, 'citationsByYear': List(Struct({'count': Int64, 'year': Int32}))})
│   ├── total: Int64
│   └── citationsByYear: List(Struct

## 2. Analysis of coverage of some relevant fields

### 2.1. Global statistics

In [5]:
total_papers = 0
with_abstract = 0
with_affiliations = 0
with_country = 0
with_citations = 0
with_references = 0
with_funding = 0

for path in tqdm(parquet_files, desc="Computing OpenAIRE global stats"):
    df = pl.read_parquet(
        path,
        columns=["abstracts", "affiliations", "citations", "references", "funding"]
    )

    n = df.height
    total_papers += n

    # abstracts: lista no vacía
    with_abstract += df.select(pl.col("abstracts").list.len().gt(0)).sum().item()

    # affiliations: lista no vacía
    with_affiliations += df.select(pl.col("affiliations").list.len().gt(0)).sum().item()

    # country en affiliations: al menos una afiliación con country no nulo
    with_country += (
        df
        .select(
            pl.col("affiliations")
              .list.eval(pl.element().struct.field("country").is_not_null())
              .list.any()
        )
        .sum()
        .item()
    )

    # citations: struct no nulo y total > 0
    with_citations += (
        df
        .select(pl.col("citations").struct.field("total").fill_null(0).gt(0))
        .sum()
        .item()
    )

    # references: struct no nulo y total > 0
    with_references += (
        df
        .select(pl.col("references").struct.field("total").fill_null(0).gt(0))
        .sum()
        .item()
    )

    # funding: struct no nulo y projectCount > 0
    with_funding += (
        df
        .select(pl.col("funding").struct.field("projectCount").fill_null(0).gt(0))
        .sum()
        .item()
    )

    del df

Computing OpenAIRE global stats: 100%|███████████████████████████████████████████████████████████████| 800/800 [02:53<00:00,  4.62it/s]


In [6]:
stats = {
    "Total papers": total_papers,
    "With abstract": with_abstract,
    "With affiliations": with_affiliations,
    "With country info": with_country,
    "With citations": with_citations,
    "With references": with_references,
    "With funding": with_funding,
}

for k, v in stats.items():
    if k == "Total papers":
        print(f"{k:20s}: {v:,}")
    else:
        pct = 100 * v / total_papers if total_papers else 0
        print(f"{k:20s}: {v:,}  ({pct:.2f}%)")

Total papers        : 140,306,502
With abstract       : 87,378,615  (62.28%)
With affiliations   : 54,564,586  (38.89%)
With country info   : 54,564,586  (38.89%)
With citations      : 40,504,109  (28.87%)
With references     : 48,806,740  (34.79%)
With funding        : 3,789,139  (2.70%)


### 2.2. Paper ids breakdown

In [7]:
total_papers = 0
pid_type_counts = Counter()

for path in tqdm(parquet_files, desc="Counting PID types (OpenAIRE)"):
    df = pl.read_parquet(path, columns=["pids"])

    total_papers += df.height

    pids_list = df["pids"].to_list()
    for pids in pids_list:
        if not pids:
            continue
        pid_types = {pid.get("type") for pid in pids if pid and pid.get("type")}
        for t in pid_types:
            pid_type_counts[t] += 1

    del df

# Mostrar resultados
print(f"Total papers: {total_papers:,}\n")

for pid_type, cnt in pid_type_counts.most_common():
    pct = 100 * cnt / total_papers
    print(f"{pid_type:30s}: {cnt:12,d}  ({pct:6.2f}%)")

Counting PID types (OpenAIRE): 100%|█████████████████████████████████████████████████████████████████| 800/800 [04:15<00:00,  3.13it/s]

Total papers: 140,306,502

Digital Object Identifier     :  116,107,931  ( 82.75%)
Microsoft Academic Graph Identifier:   65,879,022  ( 46.95%)
PubMed ID                     :   21,807,644  ( 15.54%)
Handle                        :   12,320,952  (  8.78%)
PubMed Central ID             :    7,247,351  (  5.17%)
arXiv                         :    2,496,598  (  1.78%)


### 2.3. Number of abstracts available in different subsets

In [9]:
total_counts = Counter()
with_abstract_counts = Counter()

for path in tqdm(parquet_files, desc="Counting abstracts per PID type"):
    df = pl.read_parquet(path, columns=["pids", "abstracts"])

    # lista booleana por fila: tiene abstract?
    has_abs = (df["abstracts"].list.len() > 0).to_list()

    # explota pids a lista plana
    exploded_pids = df["pids"].explode()

    # extrae tipos de PID
    pid_types = exploded_pids.struct.field("type").to_list()

    # contadores totales
    total_counts.update(pid_types)

    # ahora contamos solo los que vienen de filas con abstract
    # reconstruimos el mapeo fila -> nº de pids
    lens = df["pids"].list.len().fill_null(0).to_list()

    idx = 0
    for has, n in zip(has_abs, lens):
        if has:
            for _ in range(n):
                with_abstract_counts[pid_types[idx]] += 1
                idx += 1
        else:
            idx += n

    del df, exploded_pids

# construir tabla final
import polars as pl

rows = []
for t, total in total_counts.items():
    with_abs = with_abstract_counts.get(t, 0)
    pct = with_abs / total * 100 if total else 0
    rows.append((t, total, with_abs, pct))

stats = pl.DataFrame(rows, schema=["type", "total", "with_abstract", "pct_with_abstract"]) \
         .sort("total", descending=True)

stats

Counting abstracts per PID type: 100%|███████████████████████████████████████████████████████████████| 800/800 [03:07<00:00,  4.27it/s]
/usr/lib/python3.10/functools.py:889: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return dispatch(args[0].__class__)(*args, **kw)


type,total,with_abstract,pct_with_abstract
str,i64,i64,f64
"""Digital Object Identifier""",126262484,80989375,64.143657
"""Microsoft Academic Graph Ident…",67591332,51485724,76.172081
"""PubMed ID""",21887460,16961453,77.49393
null,15107798,5594651,37.031545
"""Handle""",14682610,9794016,66.704871
"""PubMed Central ID""",7265280,6090730,83.833383
"""arXiv""",2504924,2405933,96.048144


## 3. FOS Analysis

### 3.1. FOS categories per level

In [13]:
fos_lvl1 = Counter()
fos_lvl2 = Counter()
fos_lvl3 = Counter()
fos_lvl4 = Counter()

papers_with_fos_lvl1 = 0
papers_with_fos_lvl2 = 0
papers_with_fos_lvl3 = 0
papers_with_fos_lvl4 = 0

for path in tqdm(parquet_files, desc="Collecting FOS labels (per paper)"):
    df = pl.read_parquet(path, columns=["fos"])

    fos_list = df["fos"].to_list()

    for fos in fos_list:
        if not fos:
            continue

        # conjuntos por paper (para no contar dos veces el mismo paper)
        lvl1_set = {f.get("lvl1") for f in fos if f and f.get("lvl1")}
        lvl2_set = {f.get("lvl2") for f in fos if f and f.get("lvl2")}
        lvl3_set = {f.get("lvl3") for f in fos if f and f.get("lvl3")}
        lvl4_set = {f.get("lvl4") for f in fos if f and f.get("lvl4")}

        if lvl1_set:
            papers_with_fos_lvl1 += 1
            fos_lvl1.update(lvl1_set)

        if lvl2_set:
            papers_with_fos_lvl2 += 1
            fos_lvl2.update(lvl2_set)

        if lvl3_set:
            papers_with_fos_lvl3 += 1
            fos_lvl3.update(lvl3_set)

        if lvl4_set:
            papers_with_fos_lvl4 += 1
            fos_lvl4.update(lvl4_set)

    del df

In [21]:
def print_counter_table(title, counter, total_papers, top_n=None):
    print(f"\n{title}")
    print("-" * len(title))
    for label, cnt in (counter.most_common(top_n) if top_n else counter.most_common()):
        pct = 100 * cnt / total_papers if total_papers else 0
        print(f"{label:50s} : {cnt:12,d}  ({pct:6.2f}%)")
    print(f"\nTotal papers with FOS at this level: {total_papers:,}")

print_counter_table("FOS level 1", fos_lvl1, papers_with_fos_lvl1)
print_counter_table("FOS level 2", fos_lvl2, papers_with_fos_lvl2)
print_counter_table("FOS level 3 (top 40)", fos_lvl3, papers_with_fos_lvl3, top_n=40)
print_counter_table("FOS level 4 (top 40)", fos_lvl4, papers_with_fos_lvl4, top_n=40)


FOS level 1
-----------
medical and health sciences                        :   25,229,030  ( 45.47%)
natural sciences                                   :   15,546,779  ( 28.02%)
engineering and technology                         :   15,265,235  ( 27.51%)
social sciences                                    :    7,505,678  ( 13.53%)
agricultural and veterinary sciences               :    1,611,868  (  2.90%)
humanities and the arts                            :    1,486,563  (  2.68%)

Total papers with FOS at this level: 55,488,814

FOS level 2
-----------
clinical medicine                                  :   16,236,797  ( 29.26%)
basic medicine                                     :   10,332,864  ( 18.62%)
electrical engineering, electronic engineering, information engineering :    6,833,198  ( 12.31%)
health sciences                                    :    6,698,837  ( 12.07%)
physical sciences                                  :    5,435,048  (  9.79%)
nano-technology                  

### 3.2. FOS categories per level in PubMed

In [22]:
fos_lvl1_pubmed = Counter()
fos_lvl2_pubmed = Counter()
fos_lvl3_pubmed = Counter()
fos_lvl4_pubmed = Counter()

papers_with_fos_lvl1_pubmed = 0
papers_with_fos_lvl2_pubmed = 0
papers_with_fos_lvl3_pubmed = 0
papers_with_fos_lvl4_pubmed = 0

for path in tqdm(parquet_files, desc="Collecting FOS labels for PubMed papers"):
    df = pl.read_parquet(path, columns=["fos", "pids"])

    fos_list = df["fos"].to_list()
    pids_list = df["pids"].to_list()

    for fos, pids in zip(fos_list, pids_list):
        if not pids:
            continue

        pid_types = {pid.get("type") for pid in pids if pid and pid.get("type")}
        if "PubMed ID" not in pid_types:
            continue

        if not fos:
            continue

        lvl1_set = {f.get("lvl1") for f in fos if f and f.get("lvl1")}
        lvl2_set = {f.get("lvl2") for f in fos if f and f.get("lvl2")}
        lvl3_set = {f.get("lvl3") for f in fos if f and f.get("lvl3")}
        lvl4_set = {f.get("lvl4") for f in fos if f and f.get("lvl4")}

        if lvl1_set:
            papers_with_fos_lvl1_pubmed += 1
            fos_lvl1_pubmed.update(lvl1_set)

        if lvl2_set:
            papers_with_fos_lvl2_pubmed += 1
            fos_lvl2_pubmed.update(lvl2_set)

        if lvl3_set:
            papers_with_fos_lvl3_pubmed += 1
            fos_lvl3_pubmed.update(lvl3_set)

        if lvl4_set:
            papers_with_fos_lvl4_pubmed += 1
            fos_lvl4_pubmed.update(lvl4_set)

    del df

In [26]:
def print_counter_table(title, counter, total_papers, top_n=None):
    print(f"\n{title}")
    print("-" * len(title))
    for label, cnt in (counter.most_common(top_n) if top_n else counter.most_common()):
        pct = 100 * cnt / total_papers if total_papers else 0
        print(f"{label:50s} : {cnt:12,d}  ({pct:6.2f}%)")
    print(f"\nTotal papers with FOS at this level: {total_papers:,}")

print_counter_table("PubMed – FOS level 1", fos_lvl1_pubmed, papers_with_fos_lvl1_pubmed)
print_counter_table("PubMed – FOS level 2", fos_lvl2_pubmed, papers_with_fos_lvl2_pubmed)
print_counter_table("PubMed – FOS level 3 (top 40)", fos_lvl3_pubmed, papers_with_fos_lvl3_pubmed, top_n=40)
print_counter_table("PubMed – FOS level 4 (top 40)", fos_lvl4_pubmed, papers_with_fos_lvl4_pubmed, top_n=40)


PubMed – FOS level 1
--------------------
medical and health sciences                        :   14,317,277  ( 83.05%)
natural sciences                                   :    2,388,203  ( 13.85%)
engineering and technology                         :    1,310,851  (  7.60%)
social sciences                                    :      696,718  (  4.04%)
agricultural and veterinary sciences               :      421,526  (  2.45%)
humanities and the arts                            :       51,393  (  0.30%)

Total papers with FOS at this level: 17,240,118

PubMed – FOS level 2
--------------------
clinical medicine                                  :    8,904,198  ( 51.65%)
basic medicine                                     :    6,434,507  ( 37.32%)
health sciences                                    :    4,316,036  ( 25.03%)
chemical sciences                                  :    1,160,839  (  6.73%)
nano-technology                                    :      759,377  (  4.40%)
psychology and cog

### 3.3. FOS categories per level in arXiv

In [27]:
fos_lvl1_arxiv = Counter()
fos_lvl2_arxiv = Counter()
fos_lvl3_arxiv = Counter()
fos_lvl4_arxiv = Counter()

papers_with_fos_lvl1_arxiv = 0
papers_with_fos_lvl2_arxiv = 0
papers_with_fos_lvl3_arxiv = 0
papers_with_fos_lvl4_arxiv = 0

for path in tqdm(parquet_files, desc="Collecting FOS labels for arXiv papers"):
    df = pl.read_parquet(path, columns=["fos", "pids"])

    fos_list = df["fos"].to_list()
    pids_list = df["pids"].to_list()

    for fos, pids in zip(fos_list, pids_list):
        if not pids:
            continue

        pid_types = {pid.get("type") for pid in pids if pid and pid.get("type")}
        if "arXiv" not in pid_types:
            continue

        if not fos:
            continue

        lvl1_set = {f.get("lvl1") for f in fos if f and f.get("lvl1")}
        lvl2_set = {f.get("lvl2") for f in fos if f and f.get("lvl2")}
        lvl3_set = {f.get("lvl3") for f in fos if f and f.get("lvl3")}
        lvl4_set = {f.get("lvl4") for f in fos if f and f.get("lvl4")}

        if lvl1_set:
            papers_with_fos_lvl1_arxiv += 1
            fos_lvl1_arxiv.update(lvl1_set)

        if lvl2_set:
            papers_with_fos_lvl2_arxiv += 1
            fos_lvl2_arxiv.update(lvl2_set)

        if lvl3_set:
            papers_with_fos_lvl3_arxiv += 1
            fos_lvl3_arxiv.update(lvl3_set)

        if lvl4_set:
            papers_with_fos_lvl4_arxiv += 1
            fos_lvl4_arxiv.update(lvl4_set)

    del df

In [31]:
print_counter_table("arXiv – FOS level 1", fos_lvl1_arxiv, papers_with_fos_lvl1_arxiv)
print_counter_table("arXiv – FOS level 2", fos_lvl2_arxiv, papers_with_fos_lvl2_arxiv)
print_counter_table("arXiv – FOS level 3 (top 40)", fos_lvl3_arxiv, papers_with_fos_lvl3_arxiv, top_n=40)
print_counter_table("arXiv – FOS level 4 (top 40)", fos_lvl4_arxiv, papers_with_fos_lvl4_arxiv, top_n=40)


arXiv – FOS level 1
-------------------
natural sciences                                   :    1,463,412  ( 76.74%)
engineering and technology                         :      529,701  ( 27.78%)
medical and health sciences                        :      167,988  (  8.81%)
social sciences                                    :       56,078  (  2.94%)
humanities and the arts                            :        5,414  (  0.28%)
agricultural and veterinary sciences               :        1,347  (  0.07%)

Total papers with FOS at this level: 1,906,933

arXiv – FOS level 2
-------------------
physical sciences                                  :      970,662  ( 50.90%)
mathematics                                        :      454,296  ( 23.82%)
electrical engineering, electronic engineering, information engineering :      378,857  ( 19.87%)
computer and information sciences                  :       95,074  (  4.99%)
basic medicine                                     :       94,549  (  4.96%)
na

## 4. arXiv Dataset

### 4.1. First level categories

In [32]:
import json

arxiv_path = "../arXiv/arxiv-metadata-oai-snapshot.json"

total_arxiv_papers = 0
top_level_cats = Counter()

with open(arxiv_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading arXiv dump"):
        rec = json.loads(line)

        total_arxiv_papers += 1

        cat_str = rec.get("categories", "")
        if cat_str:
            cats = {c.split(".")[0] for c in cat_str.split()}  # nivel 1: cs, physics, math, etc.
            for c in cats:
                top_level_cats[c] += 1

print(f"\nTotal arXiv papers: {total_arxiv_papers:,}\n")

for cat, cnt in top_level_cats.most_common():
    pct = 100 * cnt / total_arxiv_papers
    print(f"{cat:15s}: {cnt:12,d}  ({pct:6.2f}%)")

Reading arXiv dump: 2957779it [00:14, 209837.50it/s]


Total arXiv papers: 2,957,779

cs             :      879,659  ( 29.74%)
math           :      737,614  ( 24.94%)
cond-mat       :      411,608  ( 13.92%)
astro-ph       :      378,288  ( 12.79%)
physics        :      297,651  ( 10.06%)
hep-ph         :      193,779  (  6.55%)
hep-th         :      179,977  (  6.08%)
quant-ph       :      174,312  (  5.89%)
stat           :      141,166  (  4.77%)
gr-qc          :      119,505  (  4.04%)
eess           :      119,396  (  4.04%)
math-ph        :       88,215  (  2.98%)
nucl-th        :       61,660  (  2.08%)
hep-ex         :       59,183  (  2.00%)
q-bio          :       53,794  (  1.82%)
nlin           :       46,218  (  1.56%)
hep-lat        :       29,940  (  1.01%)
nucl-ex        :       28,067  (  0.95%)
q-fin          :       23,858  (  0.81%)
econ           :       14,666  (  0.50%)
chao-dyn       :        2,398  (  0.08%)
q-alg          :        1,578  (  0.05%)
alg-geom       :        1,423  (  0.05%)
solv-int       :        1

### 4.2. Subcategories

In [34]:
arxiv_path = "../arXiv/arxiv-metadata-oai-snapshot.json"

TARGET_TOP = {"cs", "math", "cond-mat"}

subcat_counts = {
    "cs": Counter(),
    "math": Counter(),
    "cond-mat": Counter(),
}

with open(arxiv_path, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Counting arXiv subcategories (cs/math/cond-mat)"):
        rec = json.loads(line)
        cat_str = rec.get("categories", "")
        if not cat_str:
            continue

        cats = set(cat_str.split())
        for c in cats:
            top = c.split(".")[0]
            if top in TARGET_TOP:
                subcat_counts[top][c] += 1

Counting arXiv subcategories (cs/math/cond-mat): 2957779it [00:14, 205804.11it/s]


In [49]:
DESCRIPTIONS = {
    # Computer Science (cs)
    "cs.LG": "Machine Learning",
    "cs.CV": "Computer Vision and Pattern Recognition",
    "cs.AI": "Artificial Intelligence",
    "cs.CL": "Computation and Language (NLP)",
    "cs.IT": "Information Theory",
    "cs.RO": "Robotics",
    "cs.CR": "Cryptography and Security",
    "cs.SY": "Systems and Control",
    "cs.NA": "Numerical Analysis",
    "cs.HC": "Human-Computer Interaction",
    "cs.DS": "Data Structures and Algorithms",
    "cs.DC": "Distributed, Parallel, and Cluster Computing",
    "cs.CY": "Computers and Society",
    "cs.NI": "Networking and Internet Architecture",
    "cs.SE": "Software Engineering",
    "cs.IR": "Information Retrieval",
    "cs.SI": "Social and Information Networks",
    "cs.SD": "Sound",
    "cs.LO": "Logic in Computer Science",
    "cs.NE": "Neural and Evolutionary Computing",
    "cs.DM": "Discrete Mathematics",
    "cs.GT": "Computer Science and Game Theory",
    "cs.CC": "Computational Complexity",
    "cs.MA": "Multiagent Systems",
    "cs.DB": "Databases",
    "cs.CE": "Computational Engineering, Finance, and Science",
    "cs.PL": "Programming Languages",
    "cs.MM": "Multimedia",
    "cs.GR": "Graphics",
    "cs.CG": "Computational Geometry",
    "cs.AR": "Hardware Architecture",
    "cs.ET": "Emerging Technologies",
    "cs.DL": "Digital Libraries",
    "cs.FL": "Formal Languages and Automata Theory",
    "cs.PF": "Performance",
    "cs.SC": "Symbolic Computation",
    "cs.MS": "Mathematical Software",
    "cs.OH": "Other Computer Science",
    "cs.OS": "Operating Systems",
    "cs.GL": "General Literature",

    # Mathematics (math)
    "math.MP": "Mathematical Physics",
    "math.CO": "Combinatorics",
    "math.AP": "Analysis of PDEs",
    "math.PR": "Probability",
    "math.OC": "Optimization and Control",
    "math.AG": "Algebraic Geometry",
    "math.IT": "Information Theory",
    "math.NT": "Number Theory",
    "math.DG": "Differential Geometry",
    "math.NA": "Numerical Analysis",
    "math.DS": "Dynamical Systems",
    "math.FA": "Functional Analysis",
    "math.RT": "Representation Theory",
    "math.ST": "Statistics Theory",
    "math.GT": "Geometric Topology",
    "math.GR": "Group Theory",
    "math.CA": "Classical Analysis and ODEs",
    "math.QA": "Quantum Algebra",
    "math.RA": "Rings and Algebras",
    "math.CV": "Complex Variables",
    "math.AT": "Algebraic Topology",
    "math.LO": "Logic",
    "math.AC": "Commutative Algebra",
    "math.OA": "Operator Algebras",
    "math.MG": "Metric Geometry",
    "math.SP": "Spectral Theory",
    "math.SG": "Symplectic Geometry",
    "math.CT": "Category Theory",
    "math.KT": "K-Theory and Homology",
    "math.GN": "General Topology",
    "math.GM": "General Mathematics",
    "math.HO": "History and Overview",

    # Condensed Matter (cond-mat)
    "cond-mat.mtrl-sci": "Materials Science",
    "cond-mat.mes-hall": "Mesoscale and Nanoscale Physics",
    "cond-mat.str-el": "Strongly Correlated Electrons",
    "cond-mat.stat-mech": "Statistical Mechanics",
    "cond-mat.supr-con": "Superconductivity",
    "cond-mat.soft": "Soft Condensed Matter",
    "cond-mat.dis-nn": "Disordered Systems and Neural Networks",
    "cond-mat.quant-gas": "Quantum Gases",
    "cond-mat.other": "Other Condensed Matter",
    "cond-mat": "Condensed Matter (general / legacy)",
}

In [50]:
def print_arxiv_subcats_with_desc(title, counter, total, desc_map):
    print(f"\n{title}")
    print("-" * len(title))
    for cat, cnt in counter.most_common():
        desc = desc_map.get(cat, "—")
        pct = 100 * cnt / total if total else 0
        print(f"{cat:20s} | {desc:40s} : {cnt:10,d}  ({pct:6.2f}%)")

# Totales por macro-área
total_cs = sum(subcat_counts["cs"].values())
total_math = sum(subcat_counts["math"].values())
total_condmat = sum(subcat_counts["cond-mat"].values())

print_arxiv_subcats_with_desc(
    "arXiv – Computer Science (cs)",
    subcat_counts["cs"],
    total_cs,
    DESCRIPTIONS
)

print_arxiv_subcats_with_desc(
    "arXiv – Mathematics (math)",
    subcat_counts["math"],
    total_math,
    DESCRIPTIONS
)

print_arxiv_subcats_with_desc(
    "arXiv – Condensed Matter (cond-mat)",
    subcat_counts["cond-mat"],
    total_condmat,
    DESCRIPTIONS
)


arXiv – Computer Science (cs)
-----------------------------
cs.LG                | Machine Learning                         :    253,157  ( 18.92%)
cs.CV                | Computer Vision and Pattern Recognition  :    181,679  ( 13.58%)
cs.AI                | Artificial Intelligence                  :    162,837  ( 12.17%)
cs.CL                | Computation and Language (NLP)           :    102,369  (  7.65%)
cs.IT                | Information Theory                       :     53,115  (  3.97%)
cs.RO                | Robotics                                 :     49,304  (  3.69%)
cs.CR                | Cryptography and Security                :     45,732  (  3.42%)
cs.SY                | Systems and Control                      :     43,261  (  3.23%)
cs.NA                | Numerical Analysis                       :     32,812  (  2.45%)
cs.HC                | Human-Computer Interaction               :     28,213  (  2.11%)
cs.DS                | Data Structures and Algorithms      

### 4.3. FOS vs arXiv categories (cs.LG, cs.CV, cs.AI)

In [54]:
df_arxiv = pl.read_ndjson("../arXiv/arxiv-metadata-oai-snapshot.json", ignore_errors=True)

TARGET_CATS = ["cs.LG", "cs.CV", "cs.AI"]

df_arxiv_sel = (
    df_arxiv
    .with_columns(pl.col("categories").str.split(" ").alias("cats"))
    .filter(
        pl.col("cats")
        .list.eval(pl.element().is_in(TARGET_CATS))
        .list.any()
    )
    .select([
        pl.col("id").alias("arxiv_id"),
        pl.col("cats")
    ])
)

print("arXiv selected:", df_arxiv_sel.height)
df_arxiv_sel.head()

arXiv selected: 465109


arxiv_id,cats
str,list[str]
"""0704.0047""","[""cs.NE"", ""cs.AI""]"
"""0704.0050""","[""cs.NE"", ""cs.AI""]"
"""0704.0304""","[""cs.IT"", ""cs.AI"", … ""q-bio.PE""]"
"""0704.0671""","[""cs.IT"", ""cs.LG"", ""math.IT""]"
"""0704.0954""","[""cs.IT"", ""cs.LG"", ""math.IT""]"


In [58]:
parquet_dir = Path("/home/jarenas/Datasets/OpenAIRE/20250113")
parquet_files = sorted(parquet_dir.glob("*.parquet"))

rows = []

for path in tqdm(parquet_files, desc="Leyendo OpenAIRE (arXiv × FOS)"):
    df = pl.read_parquet(path, columns=["pids", "fos"])

    # 1) Construir DF solo con pids explotados
    df_pid_only = (
        pl.DataFrame({
            "pid": df["pids"].explode()
        })
        .drop_nulls("pid")
        .filter(pl.col("pid").struct.field("type") == "arXiv")
        .with_columns(pl.col("pid").struct.field("pid").alias("arxiv_id"))
        .select("arxiv_id")
        .drop_nulls("arxiv_id")
    )

    if df_pid_only.height == 0:
        del df
        continue

    # 2) Repetir fos para alinearlo con los pids explotados
    df_fos_only = pl.DataFrame({
        "fos": df["fos"].explode()
    })

    # ⚠️ Alinear longitudes explícitamente
    min_len = min(df_pid_only.height, df_fos_only.height)
    df_pid_only = df_pid_only.head(min_len)
    df_fos_only = df_fos_only.head(min_len)

    # 3) Combinar y explotar fos
    df_fos = (
        pl.concat([df_pid_only, df_fos_only], how="horizontal")
        .drop_nulls("fos")
        .with_columns(pl.col("fos").explode().alias("fos_item"))
        .drop_nulls("fos_item")
        .select(
            "arxiv_id",
            pl.col("fos_item").struct.field("lvl1").alias("fos_lvl1"),
            pl.col("fos_item").struct.field("lvl2").alias("fos_lvl2"),
        )
        .drop_nulls("fos_lvl1")
    )

    if df_fos.height > 0:
        rows.append(df_fos)

    del df, df_pid_only, df_fos_only, df_fos

df_openaire_fos = pl.concat(rows) if rows else pl.DataFrame(
    {"arxiv_id": [], "fos_lvl1": [], "fos_lvl2": []}
)

print("Pares OpenAIRE (arXiv × FOS):", df_openaire_fos.height)
df_openaire_fos.head()

Leyendo OpenAIRE (arXiv × FOS): 100%|████████████████████████████████████████████████████████████████| 800/800 [00:36<00:00, 21.82it/s]

Pares OpenAIRE (arXiv × FOS): 1850211


arxiv_id,fos_lvl1,fos_lvl2
str,str,str
"""http://arxiv.org/abs/2209.0011…","""natural sciences""","""biological sciences"""
"""http://arxiv.org/abs/1108.2011""","""agricultural and veterinary sc…","""agriculture, forestry, and fis…"
"""http://arxiv.org/abs/math/0512…","""medical and health sciences""","""health sciences"""
"""http://arxiv.org/abs/1807.1136…","""medical and health sciences""","""basic medicine"""
"""http://arxiv.org/abs/2410.0016…","""engineering and technology""","""chemical engineering"""


In [60]:
df_openaire_fos = df_openaire_fos.with_columns(
    pl.col("arxiv_id")
      .str.replace(r"^https?://arxiv\.org/abs/", "")
      .str.replace(r"^http://arxiv\.org/abs/", "")
      .alias("arxiv_id_clean")
)

df_openaire_fos = (
    df_openaire_fos
    .drop("arxiv_id")
    .rename({"arxiv_id_clean": "arxiv_id"})
)

df_openaire_fos.head()

fos_lvl1,fos_lvl2,arxiv_id
str,str,str
"""natural sciences""","""biological sciences""","""2209.00115"""
"""agricultural and veterinary sc…","""agriculture, forestry, and fis…","""1108.2011"""
"""medical and health sciences""","""health sciences""","""math/0512023"""
"""medical and health sciences""","""basic medicine""","""1807.11369"""
"""engineering and technology""","""chemical engineering""","""2410.00168"""


In [61]:
df_join = df_arxiv_sel.join(df_openaire_fos, on="arxiv_id", how="inner")

print("Papers cruzados arXiv–OpenAIRE:", df_join.height)
df_join.head()

Papers cruzados arXiv–OpenAIRE: 256914


arxiv_id,cats,fos_lvl1,fos_lvl2
str,list[str],str,str
"""2209.00115""","[""stat.ML"", ""cs.LG""]","""natural sciences""","""biological sciences"""
"""2207.08080""","[""cs.CV""]","""natural sciences""","""earth and related environmenta…"
"""2310.01597""","[""cs.LG"", ""math.AT""]","""medical and health sciences""","""clinical medicine"""
"""1902.02660""","[""cs.LG"", ""stat.ML""]","""medical and health sciences""","""clinical medicine"""
"""2112.15290""","[""cs.CL"", ""cs.LG""]","""medical and health sciences""","""other medical science"""


In [62]:
def pretty_dist(df, col, title):
    out = (
        df.group_by(col)
          .len()
          .sort("len", descending=True)
          .with_columns(
              (pl.col("len") / pl.col("len").sum() * 100).alias("pct")
          )
    )

    print("\n" + title)
    print("=" * len(title))
    for r in out.iter_rows():
        print(f"{r[0]:40s} : {r[1]:12,d}  ({r[2]:6.2f}%)")
    return out


dist_lvl1 = pretty_dist(
    df_join,
    "fos_lvl1",
    "FOS nivel 1 (arXiv cs.LG + cs.CV + cs.AI)"
)

dist_lvl2 = pretty_dist(
    df_join,
    "fos_lvl2",
    "FOS nivel 2 (arXiv cs.LG + cs.CV + cs.AI)"
)


FOS nivel 1 (arXiv cs.LG + cs.CV + cs.AI)
medical and health sciences              :      112,246  ( 43.69%)
natural sciences                         :       55,173  ( 21.48%)
engineering and technology               :       53,261  ( 20.73%)
social sciences                          :       26,279  ( 10.23%)
agricultural and veterinary sciences     :        5,444  (  2.12%)
humanities and the arts                  :        4,511  (  1.76%)

FOS nivel 2 (arXiv cs.LG + cs.CV + cs.AI)
clinical medicine                        :       62,258  ( 24.23%)
basic medicine                           :       28,476  ( 11.08%)
electrical engineering, electronic engineering, information engineering :       22,840  (  8.89%)
health sciences                          :       18,530  (  7.21%)
physical sciences                        :       18,363  (  7.15%)
chemical sciences                        :       14,350  (  5.59%)
nano-technology                          :       11,825  (  4.60%)
economics an